# Stage 1 — Парсинг frame-level YT8M

**Цель:** вытащить покадровые признаки `rgb` (T, 1024) и `audio` (T, 128) из `SequenceExample` — в отличие от старого пайплайна, где на видео был **один** усреднённый вектор.

На выходе — паддированные uint8-массивы длины `L=60` кадров (дробление на float делается в DataLoader), маска реальных длин и метки жанров.

**Формат YT8M frame-level:**
- `context.labels` — int64 список YT8M-меток
- `feature_lists.rgb` — T кадров, каждый 1024 байта (uint8, квантизация min=-2, max=2)
- `feature_lists.audio` — T кадров, каждый 128 байт (аналогично)
- T ∈ [1, 300], 1 кадр ≈ 1 секунда видео

In [1]:
# ============================================================
# 1.0 — Импорты и конфиг
# ============================================================
import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path
import json, time, warnings
warnings.filterwarnings('ignore')

# ── Пути ─────────────────────────────────────────────────────
BASE_DIR   = Path('../data2')
DATA_DIR   = BASE_DIR / 'video' / 'frame_validate'   # frame-level shards
OUT_DIR    = BASE_DIR / 'frame_processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Параметры ────────────────────────────────────────────────
L        = 60          # фиксированная длина последовательности (кадров = секунд)
FRAME_STRIDE = 1       # шаг между кадрами (1 — подряд, 2 — через один)
DIM_RGB  = 1024
DIM_AUD  = 128

# ── Маппинг YT8M label_id → наш жанр (из старого пайплайна) ──
LABEL_MAP = {
    0: 'Gaming',   1: 'Gaming',   27: 'Gaming',  35: 'Gaming',
    36:'Gaming',   42: 'Gaming',  81: 'Gaming',
    3: 'Music',    4:  'Music',   9:  'Music',   10: 'Music',
    13:'Music',    14: 'Music',   28: 'Music',   31: 'Music',
    33:'Music',    34: 'Music',   37: 'Music',   38: 'Music',
    40:'Music',    41: 'Music',   86: 'Music',
    5: 'Animation', 16:'Animation',
    2: 'Vehicles', 7:  'Vehicles',17: 'Vehicles',19: 'Vehicles',
    30:'Vehicles', 44: 'Vehicles',45: 'Vehicles',
    12:'Sports',   43: 'Sports',  82: 'Sports',
    15:'Animals',  18: 'Animals',
    11:'Food',     20: 'Food',    22: 'Food',
    29:'Food',     32: 'Food',
    8: 'Dance',
    6: 'Performance',
    21:'Tech',     23: 'Tech',    24: 'Tech',
    39:'Beauty',
    25:'Film',
}
GENRES    = sorted(set(LABEL_MAP.values()))
GENRE2IDX = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE = {i: g for g, i in GENRE2IDX.items()}

# Приоритет при мультилейбле — редкие выше
GENRE_PRIORITY = {
    'Beauty':0,'Film':1,'Dance':2,'Animals':3,'Sports':4,'Tech':5,
    'Performance':6,'Food':7,'Animation':8,'Vehicles':9,'Music':10,'Gaming':11,
}

print(f'DATA_DIR    : {DATA_DIR}')
print(f'OUT_DIR     : {OUT_DIR}')
print(f'L (frames)  : {L}')
print(f'Жанров      : {len(GENRES)} — {GENRES}')

I0000 00:00:1776799416.498985   68266 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776799416.529990   68266 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


DATA_DIR    : ../data2/video/frame_validate
OUT_DIR     : ../data2/frame_processed
L (frames)  : 60
Жанров      : 12 — ['Animals', 'Animation', 'Beauty', 'Dance', 'Film', 'Food', 'Gaming', 'Music', 'Performance', 'Sports', 'Tech', 'Vehicles']


I0000 00:00:1776799417.310146   68266 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# ============================================================
# 1.1 — Проверка входных файлов
# ============================================================
tfrecord_files = sorted(DATA_DIR.glob('validate*.tfrecord'))
print(f'Найдено TFRecord файлов: {len(tfrecord_files)}')
if not tfrecord_files:
    raise FileNotFoundError(
        f'Нет frame-level файлов в {DATA_DIR}.\n'
        f'Скачай командой:\n'
        f'  cd {DATA_DIR} && partition=2/frame/validate mirror=eu shard=1,200 \\\n'
        f'      uv run python ../../download.py'
    )

# Быстрая проверка первого файла — убедимся что frame-level
ds = tf.data.TFRecordDataset(str(tfrecord_files[0]))
for raw in ds.take(1):
    seq = tf.train.SequenceExample()
    seq.ParseFromString(raw.numpy())
    n_frames = len(seq.feature_lists.feature_list.get('rgb', tf.train.FeatureList()).feature)
    assert n_frames > 0, 'Feature_lists пусты — это video-level, не frame-level!'
    print(f'OK: первое видео имеет {n_frames} кадров, feature_lists = {list(seq.feature_lists.feature_list.keys())}')

Найдено TFRecord файлов: 55
OK: первое видео имеет 282 кадров, feature_lists = ['rgb', 'audio']


W0000 00:00:1776799443.927868   68266 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
I0000 00:00:1776799443.960733   68402 tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144


In [3]:
# ============================================================
# 1.2 — Парсинг: собираем padded uint8 тензоры
# ============================================================
# Храним данные как uint8 — это нативная квантизация YT8M,
# экономит 4× памяти. Dequantize делается на лету в DataLoader:
#   x = byte * (4/255) - (2 - 2/255/2)  ≈  byte*0.01569 - 1.99216
#
# Паддинг: L=60 кадров. Если T<L → справа нули, маска отметит
# реальную длину. Если T>L → берём первые L кадров (с шагом).

rgb_list    = []   # каждый элемент (L, 1024) uint8
audio_list  = []   # каждый элемент (L, 128) uint8
length_list = []   # реальная длина после паддинга/обрезания
label_list  = []   # индекс жанра
id_list     = []   # YT8M anonymized id
skipped     = 0

t0 = time.time()
for file_idx, tf_file in enumerate(tfrecord_files):
    ds = tf.data.TFRecordDataset(str(tf_file))
    for raw in ds:
        seq = tf.train.SequenceExample()
        seq.ParseFromString(raw.numpy())

        # --- метки ---
        lbls = list(seq.context.feature['labels'].int64_list.value)
        matched = [LABEL_MAP[l] for l in lbls if l in LABEL_MAP]
        if not matched:
            skipped += 1
            continue
        genre   = min(matched, key=lambda g: GENRE_PRIORITY[g])
        y_idx   = GENRE2IDX[genre]

        # --- кадры rgb ---
        rgb_frames = seq.feature_lists.feature_list['rgb'].feature
        aud_frames = seq.feature_lists.feature_list['audio'].feature
        T = min(len(rgb_frames), len(aud_frames))
        if T == 0:
            skipped += 1
            continue

        # берём первые L кадров со stride
        idxs    = list(range(0, T, FRAME_STRIDE))[:L]
        real_len = len(idxs)

        rgb_arr = np.zeros((L, DIM_RGB), dtype=np.uint8)
        aud_arr = np.zeros((L, DIM_AUD), dtype=np.uint8)
        for i, j in enumerate(idxs):
            rgb_arr[i] = np.frombuffer(rgb_frames[j].bytes_list.value[0], dtype=np.uint8)
            aud_arr[i] = np.frombuffer(aud_frames[j].bytes_list.value[0], dtype=np.uint8)

        rgb_list.append(rgb_arr)
        audio_list.append(aud_arr)
        length_list.append(real_len)
        label_list.append(y_idx)
        id_list.append(seq.context.feature['id'].bytes_list.value[0].decode('utf-8', errors='ignore'))

    if (file_idx + 1) % 5 == 0 or (file_idx + 1) == len(tfrecord_files):
        dt = time.time() - t0
        print(f'   [{file_idx+1:>3}/{len(tfrecord_files)}] '
              f'собрано {len(label_list):>6}  пропущено {skipped:>5}  '
              f'({dt:.1f}s)')

# сборка в массивы
X_rgb   = np.stack(rgb_list,   axis=0)   # (N, L, 1024) uint8
X_audio = np.stack(audio_list, axis=0)   # (N, L, 128)  uint8
lengths = np.array(length_list, dtype=np.int32)
y       = np.array(label_list,  dtype=np.int64)
ids     = np.array(id_list)

print(f'\nИтого:')
print(f'  X_rgb   : {X_rgb.shape}  dtype={X_rgb.dtype}  ~{X_rgb.nbytes/1e6:.1f} MB')
print(f'  X_audio : {X_audio.shape}  dtype={X_audio.dtype}  ~{X_audio.nbytes/1e6:.1f} MB')
print(f'  lengths : {lengths.shape}  min={lengths.min()} max={lengths.max()} mean={lengths.mean():.1f}')
print(f'  y       : {y.shape}  классов={len(np.unique(y))}')
print(f'  пропущено (нет нужных меток / пустое видео): {skipped}')

   [  5/55] собрано   1074  пропущено   343  (0.4s)
   [ 10/55] собрано   2217  пропущено   719  (0.8s)
   [ 15/55] собрано   3278  пропущено  1100  (1.1s)
   [ 20/55] собрано   4418  пропущено  1462  (1.5s)
   [ 25/55] собрано   5520  пропущено  1820  (1.9s)
   [ 30/55] собрано   6606  пропущено  2193  (2.3s)
   [ 35/55] собрано   7689  пропущено  2552  (2.6s)
   [ 40/55] собрано   8780  пропущено  2871  (3.0s)
   [ 45/55] собрано   9854  пропущено  3199  (3.4s)
   [ 50/55] собрано  10927  пропущено  3556  (3.7s)
   [ 55/55] собрано  11963  пропущено  3884  (4.1s)

Итого:
  X_rgb   : (11963, 60, 1024)  dtype=uint8  ~735.0 MB
  X_audio : (11963, 60, 128)  dtype=uint8  ~91.9 MB
  lengths : (11963,)  min=1 max=60 mean=60.0
  y       : (11963,)  классов=12
  пропущено (нет нужных меток / пустое видео): 3884


In [5]:
# ============================================================
# 1.3 — Статистика классов + отсечение малых
# ============================================================
counts = np.bincount(y, minlength=len(GENRES))
N = len(y)
print(f'Распределение классов (N={N:,}):')
print(f'  {"жанр":<14} {"N":>7}   {"%":>6}')
print('  ' + '─'*34)
for i, g in enumerate(GENRES):
    pct = counts[i] / N * 100
    bar = '█' * int(pct / 2)
    print(f'  {g:<14} {counts[i]:>7,}   {pct:>5.1f}%  {bar}')
print(f'  imbalance ratio: {counts.max()/max(counts.min(),1):.2f}×')

Распределение классов (N=11,963):
  жанр                 N        %
  ──────────────────────────────────
  Animals            497     4.2%  ██
  Animation        1,032     8.6%  ████
  Beauty             154     1.3%  
  Dance              735     6.1%  ███
  Film               252     2.1%  █
  Food               557     4.7%  ██
  Gaming           2,681    22.4%  ███████████
  Music            2,509    21.0%  ██████████
  Performance        528     4.4%  ██
  Sports             723     6.0%  ███
  Tech               375     3.1%  █
  Vehicles         1,920    16.0%  ████████
  imbalance ratio: 17.41×


In [6]:
# ============================================================
# 1.4 — Балансировка: cap per class
# ============================================================
MIN_COUNT     = max(counts.min(), 1)
MAX_PER_CLASS = min(5000, MIN_COUNT * 3)
print(f'min class={MIN_COUNT:,}  cap={MAX_PER_CLASS:,}')

rng = np.random.default_rng(42)
keep = []
for c in range(len(GENRES)):
    idx = np.where(y == c)[0]
    if len(idx) > MAX_PER_CLASS:
        idx = rng.choice(idx, MAX_PER_CLASS, replace=False)
    keep.append(idx)
keep = np.concatenate(keep)
rng.shuffle(keep)

X_rgb   = X_rgb[keep]
X_audio = X_audio[keep]
lengths = lengths[keep]
y       = y[keep]
ids     = ids[keep]

counts_bal = np.bincount(y, minlength=len(GENRES))
print(f'\nПосле балансировки: N={len(y):,}, imbalance={counts_bal.max()/max(counts_bal.min(),1):.2f}×')
print(f'  {"жанр":<14} {"до":>7} {"после":>7}')
for i, g in enumerate(GENRES):
    print(f'  {g:<14} {counts[i]:>7,} {counts_bal[i]:>7,}')

min class=154  cap=462

После балансировки: N=4,939, imbalance=3.00×
  жанр                до   после
  Animals            497     462
  Animation        1,032     462
  Beauty             154     154
  Dance              735     462
  Film               252     252
  Food               557     462
  Gaming           2,681     462
  Music            2,509     462
  Performance        528     462
  Sports             723     462
  Tech               375     375
  Vehicles         1,920     462


In [7]:
# ============================================================
# 1.5 — Сохранение
# ============================================================
np.save(OUT_DIR / 'X_rgb.npy',   X_rgb)     # (N, L, 1024) uint8
np.save(OUT_DIR / 'X_audio.npy', X_audio)   # (N, L, 128)  uint8
np.save(OUT_DIR / 'lengths.npy', lengths)
np.save(OUT_DIR / 'y.npy',       y)
np.save(OUT_DIR / 'ids.npy',     ids)

pd.DataFrame([{'label_idx':i,'category':g} for i,g in IDX2GENRE.items()]) \
  .to_csv(OUT_DIR / 'label_map.csv', index=False)

config = {
    'n_samples'    : int(len(y)),
    'n_classes'    : len(GENRES),
    'L'            : L,
    'frame_stride' : FRAME_STRIDE,
    'dim_rgb'      : DIM_RGB,
    'dim_audio'    : DIM_AUD,
    'genres'       : GENRES,
    'genre2idx'    : GENRE2IDX,
    'max_per_class': int(MAX_PER_CLASS),
    # dequantize: x = (byte + 0.5) * (max-min)/255 + min,   min=-2, max=2
    'dequant_scale': 4.0 / 255.0,
    'dequant_bias' : 4.0/(2*255) - 2.0,
}
with open(OUT_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print('=' * 55)
print('STAGE 1 — COMPLETE')
print('=' * 55)
for p in sorted(OUT_DIR.iterdir()):
    print(f'  {p.name:<20} {p.stat().st_size/1024**2:>8.2f} MB')

STAGE 1 — COMPLETE
  X_audio.npy             36.17 MB
  X_rgb.npy              289.39 MB
  config.json              0.00 MB
  ids.npy                  0.08 MB
  label_map.csv            0.00 MB
  lengths.npy              0.02 MB
  y.npy                    0.04 MB
